# 選定プレフィックスごとのフロー長分布

`flow length = packet_count`（双方向5タプルあたりパケット数）を、**Raw／Broad removed／Broad retained**で比較します。成功済みrunのCSVのみを読み、PCAP・Aguri・Scan判定を再実行しません。**このローカルMacには実データがありません。研究室PCへのSSH接続・転送も行いません。**

既存の `02_main_prefix_comparison.ipynb` と `03_prefix_deep_dive.ipynb` の所属・除外定義に合わせます。大会時の保存出力は native選定18個、所属550,289行、一意549,990フローです。これは過去の記録であり、今回の入力から再現した結果ではありません。18個を固定リストにせず、指定runの台帳と照合します。

最初に設定・manifest・選定台帳を表示します。全CSVの検査が成功するまで図を作りません。診断で停止した場合は後続セルを実行せず、設定を修正して **Kernelを再起動してRun All** してください。デモは明示指定のみです。詳細は `docs/guide/prefix-flow-length-histograms.md` を参照してください。


## 1. 入力と実行モード

`MAWI_ANALYSIS_ROOT` はコードと別のデータ保管rootでも指定できます。manifestの相対パスはそのroot基準、絶対パスは記録どおりに扱い、別ファイルを探索して代用しません。

研究室PCでの想定入口は `results/202604081400/scan_source_driven_removal/run_manifest.json`。その中の `flows`（旧互換名 `flows_csv`）、`flow_labels`、`prefixes`、`flow_prefix_membership` とflow cache manifestを確認してください。時間窓集計や送信元要約だけではこの分布は復元できません。

デモを試す場合だけ `DEMO = True` または `MAWI_FLOW_LENGTH_DEMO=1` にします。下の初期値では実データモードです。実データモードのdataset/runは明示指定が必要です。


In [ ]:
from pathlib import Path
import os
import sys
import tempfile
import json
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# Notebookをこのrepoのrootまたはnotebooks/から開いてください。
CODE_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                  if (p / 'src/mawi_global_analysis/flow_length.py').is_file()), None)
if CODE_ROOT is None:
    raise RuntimeError('このNotebookが含まれるリポジトリから開いてください')
sys.path.insert(0, str(CODE_ROOT / 'src'))
from mawi_global_analysis.flow_length import inspect_run, analyze_run, make_demo_run

DEMO = os.environ.get('MAWI_FLOW_LENGTH_DEMO', '0') == '1'
DATA_ROOT = Path(os.environ.get('MAWI_ANALYSIS_ROOT', '/home/s-sakai/mawi-global-analysis')).expanduser().resolve()
DATASET_ID = os.environ.get('MAWI_DATASET_ID', '202604081400')
RUN_NAME = os.environ.get('MAWI_RUN_NAME', 'scan_source_driven_removal')
CHUNK_SIZE = int(os.environ.get('MAWI_FLOW_LENGTH_CHUNK_SIZE', '100000'))
SCRATCH_DIR = os.environ.get('MAWI_FLOW_LENGTH_SCRATCH_DIR') or None
OUTPUT_DIR = os.environ.get('MAWI_FLOW_LENGTH_OUTPUT_DIR') or None  # 保存する場合は存在しない新規フォルダ
PREFIXES_PER_PAGE = 3
COUNT_LOG = True  # symlog: 0を残し、1より大きい度数を対数表示

# 再実行時に古い結果が残らないよう、入力検査前に破棄する。
result = None
_result_context = None

def input_context():
    return (str(DATA_ROOT), DATASET_ID, RUN_NAME, DEMO)

def require_result():
    if result is None or _result_context != input_context():
        raise RuntimeError('入力が変わったか集計に失敗しています。結果を使う前に入力検査・集計セルを再実行してください。')
if DEMO:
    demo_workspace = tempfile.TemporaryDirectory(prefix='mawi-flow-length-demo-')
    DATA_ROOT = Path(demo_workspace.name)
    DATASET_ID, RUN_NAME = make_demo_run(DATA_ROOT)
    evidence_label = 'SYNTHETIC DEMO — 実MAWIデータではありません'
else:
    if not DATASET_ID or not RUN_NAME:
        raise RuntimeError('MAWI_DATASET_ID と MAWI_RUN_NAME、または上のパラメータを明示指定してください。'
                           'このMacに実データはありません。デモはDEMO=Trueでのみ有効です。')
    evidence_label = '保存済みrun成果物の解析 — PCAP再処理なし'
print(evidence_label)
print(f'data root: {DATA_ROOT}\ndataset: {DATASET_ID}\nrun: {RUN_NAME}')

# 正確な整数度数を保存して次回のCSV全行読込を省く。既存run成果物は上書きしない。
EXACT_CACHE_DIR = Path(os.environ.get('MAWI_FLOW_LENGTH_EXACT_CACHE_DIR',
    str(DATA_ROOT / 'results' / 'flow_length_exact_cache' / DATASET_ID / RUN_NAME))).expanduser().resolve()


## 2. 設定・参照先・選定数の事前検査

成功状態、dataset、保存設定のSHA-256と現行 `scan_source_driven_removal` の設定を検査します（60秒／10秒、閾値3／2、Broad有効、IPv4、両側候補、src-or-dst所属、timeoutなし）。環境によって異なるAguri実行ファイルパスは同一性判定に用いず、Aguriオプションは比較します。

各CSVの正規スキーマ、manifest行数、flow cacheの入力SHA・fingerprint・スキーマも検査します。**manifestにCSV内容ハッシュが無い過去runでは、同じ行数・IDで内容を置換されたケースまで歴史的同一性を証明できません。** input SHAは保存記録の照合であり、PCAP現物の検証ではありません。

事前検査はヘッダーと小さいmanifest・台帳を読みます。巨大CSVの内容hashが記録されている場合の全内容検査はcache新規作成時に行います。cache再利用では入力のpath/size/mtimeとmanifestを照合します。metadataを意図的に維持した内容置換まで証明できるものではありません。


In [ ]:
result = None
_result_context = None
inspection = inspect_run(DATA_ROOT, DATASET_ID, RUN_NAME, verify_content_hashes=False)
manifest = inspection.manifest
IS_SYNTHETIC = DEMO or manifest.get('evidence_level') == 'synthetic_demo'
if IS_SYNTHETIC: evidence_label = 'SYNTHETIC DEMO — 実MAWIデータではありません'
artifact_inventory = pd.DataFrame([
    {'artifact': name, 'path': str(path),
     'manifest rows': inspection.entries[name]['row_count']}
    for name, path in inspection.paths.items()
])
display(artifact_inventory)
display(pd.DataFrame([{
    'dataset': DATASET_ID, 'run': RUN_NAME, 'status': manifest['status'],
    'config SHA-256': manifest['config']['hash'],
    'input SHA-256 (recorded)': manifest['input']['sha256'],
    'recorded Git commit': manifest.get('git_commit'),
    'selected native prefixes': len(inspection.selected_prefixes),
    'historical count': 18, 'evidence': evidence_label,
}]))
display(pd.DataFrame({'selected native prefix': inspection.selected_prefixes}))
if DEMO:
    print('デモの選定数は18と比較しません。実データ再現ではありません。')
elif DATASET_ID == '202604081400':
    print('過去18件との件数比較:', '一致' if len(inspection.selected_prefixes) == 18
          else '不一致。対象run・選定台帳を確認してください（この値は固定しません）。')
    print('件数一致だけでは、同一プレフィックス集合・同一成果物の証明にはなりません。')


## 3. 全行の照合とチャンク集計

大きいCSVを一括結合せず、一時SQLiteへ必要列だけ取り込みます。RAM使用量は主にチャンクサイズとSQLiteの32MiBキャッシュですが、ディスク使用量は全フローID・ラベル・所属に比例します。3,330万行では数GB以上になり得るため、`SCRATCH_DIR` に十分な空き容量を確保してください。SQLiteのソート用一時領域はOSの一時領域も使います。

全flow_idの一意性、flowsとlabelsのID集合一致、Strict⊆Broad、許可された観測パターンと除外ラベル、所属の一意性と存在を検査します。端点とsrc/dst_matchを照合し、native・normalized_24各集合の所属件数を全フローから確認します。ラベルの窓閾値判定自体は再実行しません。

個別prefixの総和は所属数、統合はnative所属flow_idの和集合です。片方にsrc、他方にdstが属するフローは各個別図に1回ずつ、統合図には1回だけ現れます。srcとdstが同じprefix内にある場合も、そのprefixでは1回です。normalized_24をnativeに混ぜません。

初回は整数packet_count別の度数も追加集計し、別の新規cacheへ保存します。2回目以降はmanifestと入力のpath・size・mtimeを照合し、対応する整数度数・既存summary/histogramを再利用します。粗い旧CSVからの補間は行いません。cacheがない場合は既存フローCSVの読込が一度必要です（PCAP再処理なし）。


In [ ]:
result = None
_result_context = None
summary = histogram = None
figures = []
last_report = {}
def report_progress(stage, rows):
    if stage not in last_report or rows - last_report[stage] >= 1000000:
        print(f'{stage}: {rows:,} rows')
        last_report[stage] = rows

result = analyze_run(DATA_ROOT, DATASET_ID, RUN_NAME, chunksize=CHUNK_SIZE,
                     scratch_dir=Path(SCRATCH_DIR) if SCRATCH_DIR else None,
                     progress=report_progress, exact_cache_dir=EXACT_CACHE_DIR)
_result_context = input_context()
inspection = result.inspection
manifest = inspection.manifest
artifact_inventory = pd.DataFrame([
    {'artifact': name, 'path': str(path), 'manifest rows': inspection.entries[name]['row_count']}
    for name, path in inspection.paths.items()
])
IS_SYNTHETIC = DEMO or manifest.get('evidence_level') == 'synthetic_demo'
evidence_label = 'SYNTHETIC DEMO — 実MAWIデータではありません' if IS_SYNTHETIC else '保存済みrun成果物の解析 — PCAP再処理なし' 
# 全行検査の成功後にだけ、表示用の結果を公開する。
summary = result.summary
histogram = result.histogram
prefixes = result.inspection.selected_prefixes
display(pd.DataFrame([result.audit]))
if not DEMO and DATASET_ID == '202604081400':
    display(pd.DataFrame([
        {'metric': k, 'historical recorded': v, 'this input': result.audit[k]}
        for k, v in {'selected_prefix_count': 18, 'native_membership_rows': 550289,
                     'selected_unique_flows': 549990}.items()
    ]))
print('所属行数 − 統合フロー数 =', result.audit['extra_memberships'])
print('複数native prefixに所属するフロー数 =', result.audit['multi_prefix_flows'])


## 4. 統計量

除外率の分母は各prefixのRawフロー数です。1パケット率は各群のフロー数を分母とします。中央値とp95は元の整数パケット数の度数から、NumPy/Pandasの標準と同じ線形補間で計算します。空群はNaN、小標本は分布推定として不安定なので標本数とともに読んでください。

`overall_ipv4` はIPv4全体、`selected_union` は選択先に関連する一意なフローの統合です。いずれも個別prefixの件数の単純合計ではありません。


In [ ]:
require_result()
display_summary = summary.rename(columns={
    'prefix': '対象', 'group': '群', 'flow_count': 'フロー数',
    'removal_rate': 'Rawに対する除外率', 'median': '中央値',
    'p95': 'p95', 'one_packet_rate': '群内1パケット率',
})
display(display_summary.style.format({'Rawに対する除外率': '{:.2%}',
        '中央値': '{:.3g}', 'p95': '{:.3g}', '群内1パケット率': '{:.2%}'}, na_rep='—'))


## 5. 共通ビンによる各nativeプレフィックスの比較

区間は整数半開区間 `[lower, upper)` で、**1、2、3はそれぞれ独立**、以後4–7、8–15…です。IPv4全体の最大値を含むまで上端を延ばし、すべての図・群で共通化します。長いフローを上限で切り捨てません。

横軸は区間を等間隔に並べたカテゴリです。区間幅は異なります。左は度数（設定によりsymlog）、右は群内構成比で、確率密度ではありません。除外フローはRawの部分集合なので3群を足しません。空群は凡例でn=0と示します。図内の英語表記は日本語フォントの無い研究室PCでも描画できるようにしています。


In [ ]:
require_result()
bin_labels = [str(lo) if hi == lo + 1 else f'{lo}–{hi-1}'
              for lo, hi in zip(result.bin_edges, result.bin_edges[1:])]
display(pd.DataFrame({'lower (inclusive)': result.bin_edges[:-1],
                      'upper (exclusive)': result.bin_edges[1:], 'label': bin_labels}))
colors = {'Raw': '#4477AA', 'removed': '#CC6677', 'retained': '#228833'}
figures = []

def number(value):
    return 'NA' if pd.isna(value) else f'{value:,.3g}'

def plot_page(targets, title):
    fig, axes = plt.subplots(len(targets), 2, figsize=(19, 4.5 * len(targets)), squeeze=False)
    x = np.arange(len(bin_labels))
    for row, prefix in enumerate(targets):
        stats = summary.loc[summary.prefix.eq(prefix)].set_index('group')
        rate = stats.loc['Raw', 'removal_rate']
        rate_text = 'NA' if pd.isna(rate) else f'{rate:.2%}'
        for j, group in enumerate(('Raw', 'removed', 'retained')):
            frequency = histogram.loc[histogram.prefix.eq(prefix) & histogram.group.eq(group)]
            s = stats.loc[group]
            one_text = 'NA' if pd.isna(s.one_packet_rate) else f'{s.one_packet_rate:.1%}'
            legend = (f'{group}: n={int(s.flow_count):,}, med={number(s["median"])}, '
                      f'p95={number(s.p95)}, 1pkt={one_text}')
            axes[row, 0].bar(x + (j-1)*0.26, frequency.flow_count, width=0.25,
                             color=colors[group], label=legend)
            axes[row, 1].bar(x + (j-1)*0.26, frequency.proportion*100, width=0.25,
                             color=colors[group], label=f'{group} (n={int(s.flow_count):,})')
        for col, axis in enumerate(axes[row]):
            axis.set_title(f'{prefix} | removed / Raw = {rate_text}', fontsize=11)
            axis.set_xticks(x, bin_labels, rotation=55, ha='right', fontsize=8)
            axis.set_xlabel('Packets per flow — common integer intervals')
            axis.grid(axis='y', alpha=.2)
            axis.legend(fontsize=8, loc='upper right')
            if col == 0:
                axis.set_ylabel('Flow count' + (' (symlog, linthresh=1)' if COUNT_LOG else ''))
                if COUNT_LOG: axis.set_yscale('symlog', linthresh=1)
                axis.set_ylim(bottom=0, top=max(1, axis.get_ylim()[1]))
            else:
                axis.set_ylabel('Flows within each group (%)')
                axis.set_ylim(0, 105)
    fig.suptitle(title + (' | SYNTHETIC DEMO' if IS_SYNTHETIC else f' | {DATASET_ID} / {RUN_NAME}'), fontsize=13)
    fig.tight_layout(rect=(0, 0, 1, .97))
    figures.append(fig)
    display(fig)
    plt.close(fig)

for start in range(0, len(prefixes), PREFIXES_PER_PAGE):
    plot_page(prefixes[start:start+PREFIXES_PER_PAGE], f'Native prefixes — page {start//PREFIXES_PER_PAGE+1}')
plot_page(['selected_union', 'overall_ipv4'], 'Unique selected union and overall IPv4')


## 6. 任意の保存と解釈上の注意

保存は `OUTPUT_DIR` を指定した場合のみ、存在しない新規フォルダに行います。図・統計・共通ビンの度数表と、入力manifestのSHA-256・入力設定・監査結果を保存します。元のrun、CSV、既存Notebookは変更しません。デモ出力にはSYNTHETIC DEMOを明記します。

Aguri選定先は `/24` 以上のnative範囲であり、normalized_24は別の母集団です。フローはtimeoutなしの双方向5タプル集約で、同一組の再接続を含む場合があります。Scan-likeは観測パターンに基づく表現であり、悪意の確定ではありません。MAWIの非対称観測も考慮してください。

この図は分布の構成・尾・少パケット率を比較するためのものです。分布を混ぜる操作だけから、中心極限定理による正規分布化を期待するものではありません。ソフトウェアの合成データ検証と実データによる再現・研究上の結論を分けて記録してください。


In [ ]:
require_result()
if OUTPUT_DIR:
    import hashlib
    destination = Path(OUTPUT_DIR).expanduser().resolve()
    destination.mkdir(parents=True, exist_ok=False)
    summary.to_csv(destination / 'flow_length_summary.csv', index=False)
    histogram.to_csv(destination / 'flow_length_histogram.csv', index=False)
    for i, fig in enumerate(figures, 1):
        fig.savefig(destination / f'flow_length_page_{i:02d}.png', dpi=160, bbox_inches='tight')
    source_manifest = result.inspection.manifest_path
    report = {
        'evidence': 'synthetic_demo' if IS_SYNTHETIC else 'saved_artifact_analysis_no_pcap_reprocessing',
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'dataset_id': DATASET_ID, 'run_name': RUN_NAME,
        'source_manifest': str(source_manifest),
        'source_manifest_sha256': hashlib.sha256(source_manifest.read_bytes()).hexdigest(),
        'source_config_hash': manifest['config']['hash'],
        'source_input_sha256_recorded': manifest['input']['sha256'],
        'artifact_inventory': artifact_inventory.to_dict(orient='records'),
        'bin_edges': result.bin_edges, 'quantile_method': 'linear', 'audit': result.audit,
        'limitation': 'CSV identity is not certified when source manifests omit CSV checksums',
    }
    (destination / 'analysis_manifest.json').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f'新規保存先: {destination}')
else:
    print('出力ファイルは保存していません。OUTPUT_DIR指定時のみ新規フォルダに保存します。')


## 7. 個別フロー長ヒストグラム（このNotebook内に直接表示）

台帳から取得したすべてのnativeプレフィックス、その統合群 `selected_union`、全IPv4 `overall_ipv4` を、このNotebook内に順番に表示します。18件の選定なら20対象です。対象ごとに見出し・統計表・Raw／Broad removed／Broad retainedの**独立した3枚のグラフ**を表示します。固定のプレフィックスリストは使いません。

`X_MAX=None`は全範囲、`X_MAX=100`や`200`は短小フローの拡大表示です。横軸・縦軸は線形、ビンは整数パケット数の幅1です。全範囲では長いtailのため1〜3の差が画面上で小さくなることがあります。各群の範囲外の件数・割合を必ず付記し、%の分母は範囲外を含む各群の全件数とします。

既存の比較図・CSV/PNG保存は上のセクションに残しています。以下の個別表示セルは別Notebook・リンク・ファイル・出力ディレクトリを作成せず、集計済みの整数度数を使ってその場で描画します。`X_MAX`を変えてこのセクションだけ再実行しても、フローCSVは再集計しません。


In [ ]:
require_result()
from mawi_global_analysis.flow_length_individual import plot_individual_target

_x_max = os.environ.get('MAWI_FLOW_LENGTH_X_MAX', '').strip()
X_MAX = int(_x_max) if _x_max else None
Y_AXIS = os.environ.get('MAWI_FLOW_LENGTH_Y_AXIS', 'count')  # 'count' または 'percent'
# 表示例: X_MAX = 100 / 200。ファイル保存先の設定は不要です。


### 7.1 全対象を順番に表示

設定セルを実行後、下のセルを実行してください。各対象の見出しに続いて、Rawフロー数・除外率・各群の中央値/p95/1パケット率・範囲外件数/割合と、3枚の独立した図が表示されます。対象の数は台帳に応じて自動で増減します。


In [ ]:
require_result()
individual_targets = [*result.inspection.selected_prefixes, 'selected_union', 'overall_ipv4']
for index, target in enumerate(individual_targets, start=1):
    display(Markdown(f'### 7.1.{index} {target}'))
    individual_view = plot_individual_target(result, target, x_max=X_MAX, y_axis=Y_AXIS)
    try:
        display(individual_view.distribution.statistics)
        for fig in individual_view.figures.values():
            display(fig)
    finally:
        for fig in individual_view.figures.values():
            plt.close(fig)
if DEMO:
    print('SYNTHETIC DEMO: 実データ再現ではありません。再描画のため合成入力はKernel終了まで保持します。')
